In [1]:
# pip installer
!pip install nflreadpy
!pip install pandas
!pip install polars
!pip install sklearn

In [ ]:
# Import libraries
import nflreadpy as nfl
import pandas as pd
import polars as pl


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score


In [3]:
## Load data 
seasons = list(range(2018, 2025)) # Slicing to the last 7-sevens
pbp = nfl.load_pbp(seasons=seasons)




In [ ]:
# Showing the first 5-lines
pbp.head()

In [ ]:
pbp.shape() # Printing the shape of the Polars dataframe

In [ ]:
# Getting a better understanding what EPA is
pbp.select("epa").describe()

In [21]:
pbp.select("play_type").unique() # What is inside the play_type column

In [ ]:
pbp.select("season_type").unique()
# since there is only post and regular, we keep both of them

In [24]:
# Keeping the plays where EPA is not null
pbp = pbp.filter(
    (pl.col("epa").is_not_null()) &
    (pl.col("play_type").is_in(["run","pass"]))  # removing qb_kneels, field goals and so forth
)

In [ ]:
pbp.shape # printing the new shape of the dataframe

In [ ]:
# Team-game aggregations from the pbp
## Computing offensive efficiency per team per game

team_game = ( 
    pbp.group_by(["season", "week", "game_id", "postteam"])
    .agg([
        pl.mean("epa").alias("off_epa_per_play"), # storing mean of epa as off_epa_per_play
        (pl.col("epa") > 0).mean().alias("off_success_rate"), # Storing the offensive success rate where the EPA is increased
        (pl.col("play_type") == "pass").mean().alias("off_pass_rate"), # storing the success for play type pass
        (pl.col("play_type") == "run").mean().alias("off_run_ra te"),
        pl.len().alias("off_plays") # number of offensive plays
    ])
    .rename({"posteam":"team"})
)

In [ ]:
pbp.select("game_id")

In [18]:
pbp.select("wpa").head()